In [ ]:
import json
import math
import os

import mysql.connector
import numpy as np
import pandas as pd

from decimal import Decimal
from IPython.display import display
from sshtunnel import SSHTunnelForwarder

Connect to the database

Establish connection to the main database and define query() and qdf() for SQL queries.

In [ ]:
with open('db_config.json') as f:
    DB_CFG = json.load(f)

tunnel = SSHTunnelForwarder((DB_CFG["ssh_host"], int(DB_CFG["ssh_port"])), ssh_username=DB_CFG["ssh_user"], ssh_pkey=os.path.expanduser(DB_CFG["ssh_pkey"]), remote_bind_address=("127.0.0.1", 3306))
tunnel.start()
mydb = mysql.connector.connect(host="127.0.0.1", port=tunnel.local_bind_port, user=DB_CFG["db_user"], database=DB_CFG.get("db_name", "nml"))

def query(sql):
    """Query database with SQL syntax as a string (or fstring)."""
    cursor = mydb.cursor()
    cursor.execute(sql)
    return(cursor.fetchall())

def qdf(sql, params=None):
    """query() with column names attached, parameters passed safely, and Decimal cast to float."""
    cur = mydb.cursor()
    cur.execute(sql, params or ())
    df = pd.DataFrame(cur.fetchall(), columns=[d[0] for d in cur.description])
    cur.close()
    for c in df.columns:
        if df[c].map(lambda v: isinstance(v, Decimal)).any():
            df[c] = pd.to_numeric(df[c], errors='coerce').astype(float)
    return df

query('SHOW TABLES')

`DRY_RUN = True` reports what each step would change and writes nothing. Set it to `False` and re-run to write.

`IMPORT_CSV` is only needed when building `weather_hourly` fresh from the bulk historic OpenWeather data. The `weather_daily` rebuild does not need it.

In [ ]:
DRY_RUN = True
IMPORT_CSV = False
PRECIP_ABSENT_IS_ZERO = True     # OpenWeather omits rain and snow when there is none, so NULL means 0.00
AREAS = ['waterfront', 'williambrownst', 'sudleyhouse', 'portsunlight']
SOURCES = json.dumps(["OpenWeather"])
DERIVED_DAILY = json.dumps(["Derived from weather_hourly"])
BATCH = 5000

Get hourly weather data from the OpenWeather CSV

Map the CSV's locations to the four areas in the schema, convert `dt_iso` to UTC and CSV `NaN` to NULL, and append in batches of 5,000. 

Existing rows are left as is, and the OpenWeather source label is only added where a row has none, so rows appended nightly by `owapi.py` keep their own.

In [ ]:
def clean_value(v):
    if v is None:
        return None
    if isinstance(v, float) and math.isnan(v):
        return None
    return v

def insert_batches(rows, sql, batch_size=BATCH):
    cur = mydb.cursor()
    total = 0
    for i in range(0, len(rows), batch_size):
        cur.executemany(sql, rows[i:i + batch_size])
        mydb.commit()
        total += cur.rowcount
        print(f"Inserted {total}/{len(rows)} rows...")
    cur.close()

if IMPORT_CSV:
    wdf = pd.read_csv(r'data/openweatherhistoric.csv', low_memory=False)
    area_map = {'World Museum': 'williambrownst', 'Sudley House': 'sudleyhouse',
                'Museum of Liverpool': 'waterfront', 'Lady Lever Art Gallery': 'portsunlight'}
    wdf['area'] = wdf['city_name'].map(area_map)
    wdf['datetime'] = pd.to_datetime(wdf['dt_iso'].str.replace(' UTC', '', regex=False), utc=True).dt.tz_convert(None)
    wdf = wdf.dropna(subset=['area'])
    cols = ['datetime', 'area', 'temp', 'feels_like', 'dew_point', 'visibility', 'clouds_all', 'wind_speed',
            'wind_gust', 'rain_1h', 'snow_1h', 'weather_id', 'weather_description']
    rows = [tuple(clean_value(v) for v in row) for row in wdf[cols].itertuples(index=False, name=None)]
    sql = """INSERT IGNORE INTO weather_hourly
             (dt, area, temp_c, feels_like_c, dew_point_c, visibility_m, clouds_pct, wind_speed_ms,
              wind_gust_ms, rain_1h_mm, snow_1h_mm, weather_id, weather_desc)
             VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)"""
    if DRY_RUN:
        print(f"DRY_RUN: {len(rows)} CSV rows would be offered to weather_hourly (existing rows are kept).")
    else:
        insert_batches(rows, sql)
        cur = mydb.cursor()
        cur.execute("UPDATE weather_hourly SET sources = %s WHERE sources IS NULL", (SOURCES,))
        print(f"weather_hourly rows given a source: {cur.rowcount}")
        mydb.commit(); cur.close()
else:
    print("IMPORT_CSV is False: weather_hourly is left as it is.")

rain and snow fix, no longer needed.

In [ ]:
# nulls = qdf("""SELECT area, COUNT(*) AS hours,
#                       SUM(rain_1h_mm IS NULL) AS rain_null, SUM(snow_1h_mm IS NULL) AS snow_null
#                FROM weather_hourly GROUP BY area""")
# display(nulls)

# if PRECIP_ABSENT_IS_ZERO and not DRY_RUN:
#     cur = mydb.cursor()
#     for c in ('rain_1h_mm', 'snow_1h_mm'):
#         cur.execute(f"UPDATE weather_hourly SET {c} = 0 WHERE {c} IS NULL")
#         print(f"weather_hourly.{c}: {cur.rowcount} NULLs set to 0.00")
#     mydb.commit(); cur.close()
# elif PRECIP_ABSENT_IS_ZERO:
#     print("DRY_RUN: the NULLs above would be set to 0.00.")

Aggregation rules for `weather_daily`

These are the same rules `owapi.py` applies to each night's data, so the historic rows and API appends are built the same way:

In [ ]:
LOCAL_TZ = 'Europe/London'
DEFAULT_OPEN = range(10, 17) 
MAX_MISSING_HOURS = 6           
NUMERIC = ['temp_c', 'feels_like_c', 'dew_point_c', 'visibility_m', 'clouds_pct',
           'wind_speed_ms', 'wind_gust_ms', 'rain_1h_mm', 'snow_1h_mm']
SUMMED = ['rain_1h_mm', 'snow_1h_mm']
DAILY_COLS = (['date', 'area'] + NUMERIC + ['weather_id', 'weather_desc']
              + ['oh_' + c for c in NUMERIC] + ['oh_weather_id', 'oh_weather_desc'])


def day_hours(dates):
    """Length of each local calendar day in hours: 23 or 25 on the clock-change days."""
    idx = pd.DatetimeIndex(dates)
    start = idx.tz_localize(LOCAL_TZ)
    end = (idx + pd.Timedelta(days=1)).tz_localize(LOCAL_TZ)
    return pd.Series(((end - start) / pd.Timedelta(hours=1)).astype(int), index=idx)


def summarise(frame, prefix=''):
    """Per local date: means of the numeric columns, sums of rain and snow, and the most frequent
    weather_id (ties go to the one seen first) with its first description - the same rules as
    summarise() in owapi.py."""
    g = frame.groupby('date')
    out = g[[c for c in NUMERIC if c not in SUMMED]].mean()
    sums, counts = g[SUMMED].sum(), g[SUMMED].count()
    out[SUMMED] = sums.where(counts > 0)                 # every hour NULL -> NULL, as owapi.py
    out = out[NUMERIC].round(2)

    w = frame.dropna(subset=['weather_id'])
    top = (w.groupby(['date', 'weather_id']).agg(n=('dt', 'size'), first=('dt', 'min')).reset_index()
             .sort_values(['date', 'n', 'first'], ascending=[True, False, True])
             .drop_duplicates('date'))
    desc = (w.sort_values('dt').drop_duplicates(['date', 'weather_id'])
              [['date', 'weather_id', 'weather_desc']])
    top = top.merge(desc, on=['date', 'weather_id'])[['date', 'weather_id', 'weather_desc']]
    out = out.join(top.set_index('date'))
    return out.add_prefix(prefix)


def aggregate_area(hourly, open_local, today, precip_absent_is_zero=True):
    """hourly: one area's weather_hourly rows (dt in UTC). open_local: local clock times at which any
    venue in the area recorded visits. Returns (daily frame, number of days skipped as incomplete)."""
    h = hourly.copy()
    for c in NUMERIC:
        h[c] = pd.to_numeric(h[c], errors='coerce').astype(float)
    if precip_absent_is_zero:
        h[SUMMED] = h[SUMMED].fillna(0.0)                # OpenWeather omits rain/snow when there is none
    local = pd.to_datetime(h['dt']).dt.tz_localize('UTC').dt.tz_convert(LOCAL_TZ).dt.tz_localize(None)
    h['date'] = local.dt.normalize()
    h = h[h['date'] < pd.Timestamp(today)]              # today is still incomplete, as in owapi.py

    n = h.groupby('date').size()
    need = day_hours(n.index) - MAX_MISSING_HOURS
    keep = n.index[n.to_numpy() >= need.to_numpy()]
    skipped = len(n) - len(keep)
    h = h[h['date'].isin(keep)]
    local = local.loc[h.index]

    # Opening hours: the hours with recorded visits on that day where hourly visits exist (from 2023),
    # otherwise the default local window. Both are in local clock time, so they follow the clock changes.
    open_local = pd.DatetimeIndex(pd.to_datetime(open_local))
    has_hourly = h['date'].isin(open_local.normalize())
    is_open = np.where(has_hourly, local.isin(open_local), local.dt.hour.isin(list(DEFAULT_OPEN)))

    daily = summarise(h).join(summarise(h[is_open], 'oh_'))   # a day with no open rows gets NULL oh_ values
    daily = daily.reset_index()
    return daily, skipped

`visits_hourly` is stored in local clock time, taken from the hour columns of NML's sheets, so it can be matched to local weather hours directly.

In [ ]:
vh = qdf("""SELECT DISTINCT vn.area, v.`datetime` AS local_dt
            FROM visits_hourly v JOIN venue vn ON vn.id = v.venue_id
            WHERE v.visits IS NOT NULL""")
vh['local_dt'] = pd.to_datetime(vh['local_dt'])
OPEN_BY_AREA = {a: g['local_dt'] for a, g in vh.groupby('area')}
display(vh.assign(date=vh['local_dt'].dt.normalize())
          .groupby('area').agg(first_day=('date', 'min'), last_day=('date', 'max'),
                               days_with_hourly_visits=('date', 'nunique')))

Rebuild `weather_daily` in dataframe, then compare with what is stored before  write

In [ ]:
TODAY = pd.Timestamp.now(tz=LOCAL_TZ).tz_localize(None).normalize()
PREC = {c: 1 for c in NUMERIC}
PREC.update(visibility_m=0, clouds_pct=0, rain_1h_mm=2, snow_1h_mm=2)

report, rebuilt = [], {}
for area in AREAS:
    hourly = qdf(f"""SELECT dt, {', '.join(NUMERIC)}, weather_id, weather_desc
                     FROM weather_hourly WHERE area = %s ORDER BY dt""", (area,))
    hourly['dt'] = pd.to_datetime(hourly['dt'])
    new, skipped = aggregate_area(hourly, OPEN_BY_AREA.get(area, []), TODAY, PRECIP_ABSENT_IS_ZERO)
    new['area'] = area
    rebuilt[area] = new

    old = qdf("SELECT * FROM weather_daily WHERE area = %s", (area,))
    old['date'] = pd.to_datetime(old['date'])
    cmp = new.merge(old, on='date', how='left', suffixes=('', '_old'), indicator=True)
    stored = cmp['_merge'].eq('both')

    changed = pd.Series(False, index=cmp.index)
    for c in NUMERIC + ['oh_' + c for c in NUMERIC]:
        tol = 10 ** -PREC[c.removeprefix('oh_')] + 1e-9
        same = (cmp[c] - cmp[c + '_old']).abs().le(tol) | (cmp[c].isna() & cmp[c + '_old'].isna())
        changed |= ~same
    for c in ('weather_id', 'oh_weather_id'):
        changed |= ~(cmp[c].eq(cmp[c + '_old']) | (cmp[c].isna() & cmp[c + '_old'].isna()))

    report.append({'area': area, 'rebuilt_days': len(new), 'new_days': int((~stored).sum()),
                   'changed_days': int((changed & stored).sum()),
                   'oh_weather_changed': int((~(cmp['oh_weather_id'].eq(cmp['oh_weather_id_old'])
                                                | (cmp['oh_weather_id'].isna() & cmp['oh_weather_id_old'].isna()))
                                              & stored).sum()),
                   'mean_abs_change_oh_temp_c': round(float((cmp['oh_temp_c'] - cmp['oh_temp_c_old']).abs().mean()), 3),
                   'skipped_incomplete': skipped,
                   'stored_days_not_rebuilt': len(set(old['date']) - set(new['date']))})

display(pd.DataFrame(report).set_index('area'))

UPSERT = (f"INSERT INTO weather_daily ({', '.join(DAILY_COLS + ['sources'])}) "
          f"VALUES ({', '.join(['%s'] * (len(DAILY_COLS) + 1))}) ON DUPLICATE KEY UPDATE "
          + ', '.join(f"{c}=VALUES({c})" for c in DAILY_COLS[2:] + ['sources']))

def to_db(v, col):
    if v is None or (isinstance(v, float) and math.isnan(v)):
        return None
    if isinstance(v, pd.Timestamp):
        return v.date()
    if col.endswith('weather_id'):
        return int(v)
    if isinstance(v, (np.floating, np.integer)):
        return v.item()
    return v

if DRY_RUN:
    print("DRY_RUN: nothing written. Set DRY_RUN = False and re-run this cell to write weather_daily.")
else:
    for area, new in rebuilt.items():
        rows = [tuple(to_db(v, c) for v, c in zip(r, DAILY_COLS)) + (DERIVED_DAILY,)
                for r in new[DAILY_COLS].itertuples(index=False, name=None)]
        cur = mydb.cursor()
        for i in range(0, len(rows), BATCH):
            cur.executemany(UPSERT, rows[i:i + BATCH])
            mydb.commit()
        cur.close()
        print(f"{area}: {len(rows)} days written to weather_daily")